# BrownDye2: PQR/APBS preparation

Prepare a docked complex pose for APBS and later BrownDye2 setup. The primary BrownDye rigid body in this notebook is the combined complex; protein-only and ligand-only files are optional inspection outputs.

**This notebook**: prepare protein and ligand inputs from a docked PDB.
1. Fix protein with PDBFixer (add missing atoms, protonate at target pH)
2. Assign ligand bond orders from SMILES, write SDF for antechamber
3. Write ligand metadata consumed by the complex preparation script

**Shell scripts**:
1. `complex/prep.sh` -- assign Amber/GAFF2 charges and radii, write complex and component PQRs
2. `complex/run_apbs.sh` -- solve APBS for `complex.pqr` by default
3. `substrate/prep.sh` and `substrate/run_apbs.sh` -- prepare the substrate protein body
4. `bdprep.sh` and `bdrun.sh` -- run BrownDye using complex and substrate results

In [ ]:
from pathlib import Path

from Bio.PDB import PDBIO, PDBParser
from rdkit import Chem

from mdpp.prep import ChainSelect, assign_topology, fix_pdb, run_propka

COMPLEX_PDB = Path("complex.pdb")
TMP_ROOT = Path("tmp")
STEP_DIR = TMP_ROOT / "complex" / "prep"
INTERMEDIATE_DIR = STEP_DIR / "intermediate"
STEP_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

# Canonical SMILES of the ligand (used to assign bond orders)
LIGAND_SMILES = r"Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P]([O-])(=O)OC(=O)CCCC[C@@H]4SC[C@@H]5NC(=O)N[C@H]45)[C@@H](O)[C@H]3O"
PROTEIN_CHAIN_ID = "A"
LIGAND_CHAIN_ID = "B"
PH = 7.4

## Step 1: Verify protonation and fix protein

Extract the protein chain, run PropKa as an explicit pKa/protonation-state check at the target pH, then add missing residues/atoms/hydrogens via PDBFixer.

In [ ]:
parser = PDBParser(QUIET=True)
structure = parser.get_structure("complex", str(COMPLEX_PDB))
model = structure[0]
chains = {chain.id: chain for chain in model}

pdb_io = PDBIO()
pdb_io.set_structure(structure)

# Extract protein chain into this step's intermediate folder.
protein_pdb = INTERMEDIATE_DIR / "protein.pdb"
pdb_io.save(str(protein_pdb), ChainSelect(PROTEIN_CHAIN_ID))

# Verify protonation with PropKa before PDBFixer assigns hydrogens.
propka_result = run_propka(protein_pdb)
nonstandard = propka_result.get_nonstandard(PH)
propka_report = STEP_DIR / "protein_propka.tsv"
with propka_report.open("w") as f:
    f.write(
        "residue_type\tres_num\tchain_id\tpka\tmodel_pka\tpropka_protonated\tmodel_protonated\n"
    )
    for residue in propka_result.residues:
        f.write(
            f"{residue.residue_type}\t{residue.res_num}\t{residue.chain_id}\t"
            f"{residue.pka:.3f}\t{residue.model_pka:.3f}\t"
            f"{residue.is_protonated_at(PH)}\t{residue.is_default_protonated_at(PH)}\n"
        )

print(f"PropKa residues checked: {len(propka_result.residues)} -> {propka_report}")
if nonstandard:
    print(f"PropKa predicts {len(nonstandard)} non-standard protonation state(s) at pH {PH}:")
    for residue in nonstandard:
        print(
            f"  {residue.label}: pKa={residue.pka:.2f}, model={residue.model_pka:.2f}, "
            f"PropKa={'protonated' if residue.is_protonated_at(PH) else 'deprotonated'}, "
            f"model={'protonated' if residue.is_default_protonated_at(PH) else 'deprotonated'}"
        )
else:
    print(f"PropKa agrees with model-pKa defaults for all titratable residues at pH {PH}.")

protein_fixed_pdb = STEP_DIR / "protein_fixed.pdb"
fix_pdb(protein_pdb, protein_fixed_pdb, pH=PH)
print(f"Fixed protein -> {protein_fixed_pdb}")

## Step 2: Assign ligand topology

Extract ligand chain, assign bond orders from a SMILES template, and write an SDF
for antechamber. The SMILES is needed because PDB files lack bond-order information.

In [ ]:
# Extract ligand chain to the step intermediate folder for RDKit parsing.
ligand_pdb = INTERMEDIATE_DIR / "ligand.pdb"
pdb_io.save(str(ligand_pdb), ChainSelect(LIGAND_CHAIN_ID))

# Assign bond orders from SMILES template
template_mol = Chem.MolFromSmiles(LIGAND_SMILES, sanitize=True)
ligand_net_charge = Chem.GetFormalCharge(template_mol)
print(f"Ligand net charge: {ligand_net_charge}")

mol = Chem.MolFromPDBFile(str(ligand_pdb), sanitize=True, removeHs=True)
mol_assigned = assign_topology(mol, template_mol)

# Write SDF for antechamber and metadata for the shell scripts
lig_resnames = sorted({res.get_resname().strip() for res in chains[LIGAND_CHAIN_ID].get_residues()})
lig_resname = lig_resnames[0]
mol_assigned.SetProp("_Name", lig_resname)

ligand_sdf = STEP_DIR / "ligand.sdf"
with Chem.SDWriter(str(ligand_sdf)) as w:
    w.write(mol_assigned)

(STEP_DIR / "ligand_charge.txt").write_text(f"{ligand_net_charge}\n")
(STEP_DIR / "ligand_resname.txt").write_text(f"{lig_resname}\n")
print(f"Ligand SDF -> {ligand_sdf}")
print(f"Ligand metadata -> {STEP_DIR / 'ligand_charge.txt'}, {STEP_DIR / 'ligand_resname.txt'}")

## Next: run the shell workflow

The notebook produced `tmp/complex/prep/protein_fixed.pdb`, `tmp/complex/prep/ligand.sdf`, and ligand metadata. Each following script writes its main outputs to `tmp/<body>/<step>/` and keeps transient tool files in `tmp/<body>/<step>/intermediate/`.

```bash
conda activate ambertools
cd examples/browndye
bash complex/prep.sh
bash complex/run_apbs.sh
bash substrate/prep.sh
bash substrate/run_apbs.sh
bash bdprep.sh
```

`complex/run_apbs.sh` computes the complex map by default. Component maps are optional diagnostics:

```bash
PQR_STEMS="complex protein ligand" bash complex/run_apbs.sh
```

Key outputs in `tmp/`:

| File | Description |
|------|-------------|
| `complex/prep/protein_fixed.pdb`, `complex/prep/ligand.sdf` | Notebook-generated complex prep inputs |
| `complex/ambertools/complex.pqr` | Primary complex BrownDye body |
| `complex/ambertools/protein.pqr`, `complex/ambertools/ligand.pqr` | Optional component inspection PQRs |
| `complex/apbs/complex.dx` | Primary APBS electrostatic map for the complex |
| `substrate/pdb2pqr/substrate.pqr` | Protein-only substrate PQR from PDB2PQR |
| `substrate/apbs/substrate.dx` | APBS electrostatic map for the substrate |
| `bdprep/complex_substrate_simulation.xml` | BrownDye simulation input generated from complex and substrate |
| `bdrun/results.xml`, `bdrun/rate_constant.txt` | BrownDye simulation results and rate estimate |